# Kharita : parcours en six tâches

Les valeurs manquantes sont conservées, jamais transformées automatiquement en zéro. Les modèles utilisent une séparation des communes et les caractéristiques antérieures à la cible.

## Analyse exploratoire

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"Data/resultats-elections-communales-maroc-2015-2021.csv").exists())
sys.path.insert(0,str(ROOT/"Script"))
from kharita_pipeline import load_raw,clean_data,make_comparison,build_features,save_clean_outputs
import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns
sns.set_theme(style="whitegrid")
raw=load_raw(); df,quality,party_cols=clean_data(raw)
print(f"{len(df):,} lignes; années {quality['years']}")

## Analyse exploratoire

Le notebook suit les choix de nettoyage documentés; les champs absents restent manquants et les analyses restent descriptives.

In [ ]:
display(df.groupby("year").turnout_pct.agg(["count","mean","median","std","min","max"]))
fig,ax=plt.subplots(figsize=(8,5)); sns.boxplot(data=df,x="year",y="turnout_pct",ax=ax); ax.set(title="Participation par scrutin",ylabel="Participation (%)"); plt.show()
regional=df.groupby(["year","region_name"],as_index=False).turnout_pct.mean()
fig,ax=plt.subplots(figsize=(13,6)); sns.barplot(data=regional,x="region_name",y="turnout_pct",hue="year",ax=ax); ax.tick_params(axis="x",rotation=70); plt.tight_layout(); plt.show()
display(df.groupby(["year","lead_party"]).size().rename("communes_en_tete").reset_index().sort_values(["year","communes_en_tete"],ascending=[True,False]).groupby("year").head(8))
# Distribution des voix par parti, agrégée sur les champs disponibles par année.
party_totals=df.groupby("year")[party_cols].sum(min_count=1).T
party_totals.columns=party_totals.columns.astype(str)
display(party_totals)
top=party_totals.fillna(0).sum(axis=1).nlargest(10).index
plot=party_totals.loc[top].reset_index(names="parti").melt(id_vars="parti",var_name="année",value_name="voix")
fig,ax=plt.subplots(figsize=(12,6)); sns.barplot(data=plot,x="parti",y="voix",hue="année",ax=ax); ax.tick_params(axis="x",rotation=45); ax.set(title="Voix renseignées par parti (top 10)",ylabel="Voix"); plt.tight_layout(); plt.show()

c=df[["turnout_pct","votes_total","seats_total","lead_share_pct","party_concentration","winning_margin"]].corr(); sns.heatmap(c,annot=True,cmap="vlag",center=0); plt.title("Corrélations descriptives, sans causalité"); plt.show()